# Task B – SOC Automation Walkthrough

This notebook demonstrates how to interact with the enriched SOC automation pipeline directly from Python. It reuses the trained model bundle from Task A, exposes helper functions to classify messages, extracts indicators of compromise (IOCs), and surfaces analyst-ready context such as risk levels, rationale, and recommended actions.

## 1. Environment bootstrap

Add the repository to the Python path and verify that the Task A artifacts are present. Run the Task A notebook first if the bundle is missing.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the project root so that 'src/' is available.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

ARTIFACT_BUNDLE = PROJECT_ROOT / "artifacts"
MODEL_PATH = ARTIFACT_BUNDLE / "best_model.joblib"

if not MODEL_PATH.exists():
    raise FileNotFoundError("best_model.joblib not found. Execute Task A to train and export the bundle.")

print(f"Using artifacts from: {ARTIFACT_BUNDLE}")

## 2. Helper functions for analyst-style scoring

The helper below feeds raw email content into the SOC pipeline, extracts IOCs, summarises them, and bundles the enriched response for downstream analysis.

In [ ]:
from dataclasses import dataclass
from typing import Dict

from src.task_b_soc.pipeline import classify_text, reload_model
from src.task_b_soc.enrich import extract_iocs, summarize_iocs

reload_model()  # ensure a fresh model load

@dataclass
class SOCInference:
    subject: str
    body: str
    is_html: bool = False

    def run(self) -> Dict:
        combined = f"{self.subject}

{self.body}".strip()
        model_output = classify_text(combined, is_html=self.is_html)
        iocs = extract_iocs(combined)
        ioc_summary = summarize_iocs(iocs)
        return {
            "subject": self.subject,
            "risk_level": model_output.get("risk_level"),
            "label": model_output.get("label"),
            "probability": model_output.get("score"),
            "confidence": model_output.get("confidence"),
            "rationale": model_output.get("explanations", {}).get("rationale"),
            "top_terms": model_output.get("explanations", {}).get("supporting_terms"),
            "recommendations": model_output.get("recommendations", []),
            "ioc_summary": ioc_summary,
            "iocs": iocs,
        }

## 3. Score a few representative messages

Feel free to modify the examples to experiment with different subjects, HTML bodies, or more complex phishing content.

In [ ]:
examples = [
    {
        "subject": "Payroll update",
        "body": "Hi team, please review the attached payroll spreadsheet before Friday. Thanks!",
    },
    {
        "subject": "URGENT: Reset your account password now",
        "body": "Your account has been compromised. Visit http://secure-login-example.com immediately to verify your credentials or your account will be terminated.",
    },
]

outputs = []
for example in examples:
    inference = SOCInference(**example)
    enriched = inference.run()
    outputs.append(enriched)

outputs

## 4. Batch triage from CSV exports

When analysts upload CSV exports from mailboxes (with `subject`, `body`, and optional `body_is_html` columns), this routine returns a dataframe enriched with the SOC context and IOC summaries.

In [ ]:
import pandas as pd

def score_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for row in df.itertuples(index=False):
        subject = getattr(row, "subject", "")
        body = getattr(row, "body", "")
        is_html = bool(getattr(row, "body_is_html", False))
        inference = SOCInference(subject=subject, body=body, is_html=is_html)
        enriched = inference.run()
        record = {
            "subject": enriched["subject"],
            "label": enriched["label"],
            "probability": enriched["probability"],
            "risk_level": enriched["risk_level"],
            "confidence": enriched["confidence"],
            "ioc_total": enriched["ioc_summary"].get("total", 0),
            "recommendations": " | ".join(enriched["recommendations"]),
        }
        records.append(record)
    return pd.DataFrame(records)

sample_df = pd.DataFrame(examples)
score_dataframe(sample_df)

## 5. Persist enriched results for the API/UI

The FastAPI service reads the `artifacts/` directory directly. Optionally, export a convenience JSON file for downstream integrations.

In [ ]:
import json

OUTPUT_JSON = PROJECT_ROOT / "reports" / "soc_latest_batch.json"
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_JSON.open("w", encoding="utf-8") as fh:
    json.dump(outputs, fh, indent=2)

print(f"Saved sample SOC batch output to {OUTPUT_JSON}")